# Create co-occurence graph

Steps applied in this notebook

1. Software mentions curated as "not a software" were removed.
2. Mentions without a "mapped to software" entry (i.e., marked as "not\_disambiguated") were normalized to their own software mention.
3. Software mappings with NaN and None values were removed.
4. Only papers that were classified using the open alex dataset were used (see notebook `01_publication_classification` on how these results were obtained). 
5. Papers in which software did not cooccur were removed
6. A cooccurence graph was generated from the remaining software mentions. 

Described in the paper as follows:

For preprocessing, three steps were applied: (1) software mentions curated as "not a software" were removed; (2) NaN and None values were removed; and (3) rows that were unable to disambiguated (i.e., marked as "not\_disambiguated" for "mapped\_to\_software") were normalized to their own software mention. This is the reason why the 'Number of unique disambiguated mentions' row of the 'valid\_doi' column has greatly increased from 97,662 to 891,113.

The co-occurrence graph was constructed by grouping the CZI data by DOI and aggregating on the "mapped to software" field, producing a per-DOI list of software. These lists were then sorted, and a hashmap was used to track all observed software pairs and their frequencies. Iterating over all papers, every pairwise tuple and its frequency was recorded, and this hashmap was subsequently used to construct the co-occurrence graph. Each software vertex was then annotated with the DOIs in which that software was mentioned, along with the total number of papers. Software vertices that did not connect to any other software were removed.

1. [Notebook Setup](#setup)
2. [Cleaning](#cleaning)
    1. [Software Mapping](#normalize)
    2. [Metrics of the cleaning process](#metrics)
    3. [Apply cleaning](#apply_clean)
    4. [Save doi set](#doi_set)
3. [Create cooccurence graph](#graph)
4. [Save cooccurence graph](#save)
5. [How to Load cooccurence graph](#load)

<a id='setup'></a>

## Notebook setup
required packages and directory specifications
The notebook assumes that the dataset files are stored in a folder `data` that sits as the same level as the `notebooks` directory. The assumed directory structure is the following:

- `notebooks`
-  `data` 
    - `occurence_graph`
        - metrics.txt
        - agg_df.csv.gz
        - graph_joblib.pkl.gz

In [2]:
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from pathlib import Path
import plotly.express as px
from itertools import combinations
from collections import defaultdict
import igraph as ig
import gzip
from pathlib import Path
import joblib
import ast
import leidenalg
import ast
from sklearn.metrics.cluster import normalized_mutual_info_score
import clustering_mi as cmi

pd.set_option('max_colwidth', 1000)

In [ ]:
# Obtained from 01_publication_classification
ROOT_DISAMBIGUATED_CLASSIFICATION_DIR = '../data/open_alex_matches/'
matches_df = pd.read_csv(ROOT_DISAMBIGUATED_CLASSIFICATION_DIR + '/matches.csv.gz', engine = 'python', compression="gzip")

In [4]:
# version 2025aug Oct 29, 2025. zenedo: 10.5281/zenodo.17442025
# retrieving classification tables
ROOT_CLASSIFICATION_DIR = '../data/classification_openalex_2025/'
macro_cluster_main_field_path = ROOT_CLASSIFICATION_DIR + 'macro_cluster_main_field.tsv'
main_field_path = ROOT_CLASSIFICATION_DIR + 'main_field.tsv'

macro_cluster_main_field_df = pd.read_csv(macro_cluster_main_field_path, sep = '\\t', engine = 'python')
main_field_df = pd.read_csv(main_field_path, sep = '\\t', engine = 'python')

In [5]:
ROOT_LEIDEN_MICRO_DIR = Path("../data/leiden_micro/")

In [ ]:
ROOT_OCCURRENCE_GRAPH = Path("../data/occurence_graph/")

agg_df = pd.read_csv(ROOT_OCCURRENCE_GRAPH / 'agg_df.csv.gz', compression="gzip")

# convert software_list to a list
agg_df['software_list'] = agg_df['software_list'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [ ]:
agg_df = agg_df.merge(
    matches_df[['doi', 'micro_cluster_id']],
    on='doi',
    how='left'
)
agg_df

,doi,software_list,micro_cluster_id
0,10.1001/jamacardio.2018.2866,"[Stata, R]",261
1,10.1001/jamacardio.2018.3965,"[BaseSpace, FastQC, UMI-tools, FreeBayes, SnpEff, SnpSift, R, language, SPSS]",309
2,10.1001/jamacardio.2019.3891,"[SEER*Stat, R]",235
3,10.1001/jamacardio.2019.5954,"[PLINK, SnpEff, SnpSift, MetaSVM, PROVEAN, MutationTaster, PolyPhen, MatchIt, R, Survival package]",394
4,10.1001/jamacardio.2020.0246,"[PLINK, R, SAS]",387
...,...,...,...
748700,10.7717/peerj.9993,"[PELOD, R, Prism, PRISMIII, PRISMIV, PIM]",218
748701,10.7717/peerj.9994,"[DESeq2, TargetScan, DIANA-miRPath, Read Archive, Cutadapt, miRDeep2, Kalign, NumPy, Pandas, SciPy stats, UGENE, miRDB]",62
748702,10.7717/peerj.9996,"[miRanda, Cytoscape, CIBERSORT, R, survminer, survival]",1297
748703,10.7717/peerj.9997,"[QGIS, ArcMap, R, R package SSDM]",659


<a id='graph'></a>
# Create cooccurence graph

A network is created using the unique mentions (i.e., the software) as vertices and the edge between two papers indicates they have appeared together in the same paper. The weight indicates how often these two software cooccur across all papers.
Every software also has every paper it occurs in attached in with the `dois` label. Thus calling the `dois` attribute of a software node returns all the papers that software was found in.  

Every node has the `doi_count` attribute which is simply the number of papers it occured in. 

I can copy this all from previous cooccurence step

In [ ]:
cluster_graphs = {}

# create a per micro_cluster co-occurence graph
for cluster_id, group in agg_df.groupby("micro_cluster_id"):
    
    # edges
    cooccurrence = defaultdict(int)
    for software_list in group['software_list']:
        for pair in combinations(sorted(software_list), 2):
            cooccurrence[pair] += 1

    edges = [(s1, s2, freq) for (s1, s2), freq in cooccurrence.items()]

    # software vertices
    all_software = list({s for software_list in group['software_list'] for s in software_list})
    
    g = ig.Graph()
    g.add_vertices(all_software)
    
    if edges:  # guard for clusters with only 1 software (no pairs)
        g.add_edges([(s1, s2) for s1, s2, _ in edges])
        g.es['weight'] = [freq for _, _, freq in edges]

    # attach vertex metadata
    software_dois = defaultdict(list)
    for _, row in group.iterrows():
        for software in row['software_list']:
            software_dois[software].append(row['doi'])

    g.vs['dois'] = [software_dois[v['name']] for v in g.vs]
    g.vs['doi_count'] = [len(set(software_dois[v['name']])) for v in g.vs]

    cluster_graphs[cluster_id] = g

In [ ]:
cluster_partitions = {}
cluster_stability = {}
seed = 42069
n_runs = 3
rng = np.random.default_rng(seed)
seeds = rng.integers(0, 100000, size=n_runs).tolist()

def compute_nmi_stability(graph, reference_partition, seeds):
    """Compare a reference partition against n random seeds using NMI and CMI."""
    reference_membership = reference_partition.membership
    nmi_scores = []
    cmi_scores = []
    for seed in seeds:
        p = leidenalg.find_partition(
            graph,
            leidenalg.ModularityVertexPartition,
            n_iterations=-1,
            seed=seed,
            weights='weight',
        )
        nmi_scores.append(normalized_mutual_info_score(reference_membership, p.membership))
        cmi_scores.append(cmi.normalized_mutual_information(reference_membership, p.membership))
    return {
        "nmi_mean": np.mean(nmi_scores),
        "nmi_std":  np.std(nmi_scores),
        "nmi_min":  np.min(nmi_scores),
        "nmi_max":  np.max(nmi_scores),
        "cmi_mean": np.mean(cmi_scores),
        "cmi_std":  np.std(cmi_scores),
        "cmi_min":  np.min(cmi_scores),
        "cmi_max":  np.max(cmi_scores),
    }

for cluster_id, g in tqdm(cluster_graphs.items(), total=len(cluster_graphs), desc="Partitioning clusters"):
    # graphs with 1 node or no edges are not intersting :(
    if g.vcount() < 2 or g.ecount() == 0:
        cluster_partitions[cluster_id] = None
        cluster_stability[cluster_id] = None
        continue
    
    partition = leidenalg.find_partition(
        g,
        leidenalg.ModularityVertexPartition,
        n_iterations=-1,
        seed=seed,
        weights='weight',
    )
    cluster_partitions[cluster_id] = partition
    cluster_stability[cluster_id] = compute_nmi_stability(g, partition, seeds)


Partitioning clusters: 100%|██████████| 3305/3305 [12:35<00:00,  4.38it/s]


In [ ]:
len(cluster_partitions.keys())

3305

In [ ]:
from collections import defaultdict
import pandas as pd


class CommunityAnalyzer:
    def __init__(
        self,
        macro_cluster_main_field_df: pd.DataFrame,
        main_field_df: pd.DataFrame,
        matches_df: pd.DataFrame,
    ):
        self.matches_df = matches_df

        # early compute: macro_cluster_id -> set(main_field)
        self.macro_to_main_fields = (
            macro_cluster_main_field_df
            .merge(main_field_df, on='main_field_id', how='left')
            .groupby('macro_cluster_id')['main_field']
            .apply(set)
            .to_dict()
        )

    def analyze_subgraph(self, subgraph, community_index: int):
        # software frequency (node level) 
        software_freq = {v['name']: v['doi_count'] for v in subgraph.vs}

        # Collect all unique dois in this community from node metadata
        doi_freq = defaultdict(int)
        for v in subgraph.vs:
            for doi in v['dois']:
                doi_freq[doi] += 1
        community_dois = set(doi_freq.keys())

        # look up cluster ids for all dois in this community from matches_df
        community_matches = self.matches_df[self.matches_df['doi'].isin(community_dois)]

        macro_dois = defaultdict(set)
        meso_dois = defaultdict(set)
        micro_dois = defaultdict(set)
        field_dois = defaultdict(set)

        for _, row in community_matches.iterrows():
            doi = row['doi']

            macro_id = row['macro_cluster_id']
            meso_id = row['meso_cluster_id']
            micro_id = row['micro_cluster_id']

            macro_dois[macro_id].add(doi)
            meso_dois[meso_id].add(doi)
            micro_dois[micro_id].add(doi)

            for field in self.macro_to_main_fields.get(macro_id, set()):
                field_dois[field].add(doi)
                                                                               # Pretty outline of the types and meaning of the columns :p
                                                                               # The community data has been saved
                                                                               # These are basically of hashmaps for the analyses  
        return {
            'community': community_index,                                      # int: index of the community in the partition
            'community_size': len(software_freq),                              # int: number of software nodes in the community
            'doi_count': len(community_dois),                                  # int: number of unique DOIs across all nodes in the community
            'software_freq': dict(software_freq),                              # dict[str, int]: software name    -> number of DOIs it appears in
            'doi_freq': dict(doi_freq),                                        # dict[str, int]: DOI              -> number of software mentions that cite it
            'macro_cluster_freq': {k: len(v) for k, v in macro_dois.items()},  # dict[int, int]: macro cluster id -> number of unique DOIs mapped to it
            'meso_cluster_freq': {k: len(v) for k, v in meso_dois.items()},    # dict[int, int]: meso  cluster id -> number of unique DOIs mapped to it
            'micro_cluster_freq': {k: len(v) for k, v in micro_dois.items()},  # dict[int, int]: micro cluster id -> number of unique DOIs mapped to it
            'main_field_freq': {k: len(v) for k, v in field_dois.items()},     # dict[str, int]: main  field name -> number of unique DOIs mapped to it
        }

    def run(self, partition):
        rows = []
        for i, subgraph in enumerate(partition.subgraphs()):
            rows.append(self.analyze_subgraph(subgraph, i))
        return pd.DataFrame(rows).sort_values('community')

In [ ]:
all_community_summaries = []
global_community_index = 0

for cluster_id, g in cluster_graphs.items():
    partition = cluster_partitions.get(cluster_id)
    if partition is None:
        continue

    stability = cluster_stability.get(cluster_id)

    analyzer = CommunityAnalyzer(
        macro_cluster_main_field_df=macro_cluster_main_field_df,
        main_field_df=main_field_df,
        matches_df=matches_df,
    )

    for i, subgraph in enumerate(partition.subgraphs()):
        row = analyzer.analyze_subgraph(subgraph, community_index=global_community_index)
        row["micro_cluster_id"] = cluster_id
        row["community_index"] = i
        row["modularity"] = partition.modularity
        row["nmi_mean"] = stability["nmi_mean"]
        row["nmi_std"]  = stability["nmi_std"]
        row["nmi_min"]  = stability["nmi_min"]
        row["nmi_max"]  = stability["nmi_max"]
        row["cmi_mean"] = stability["cmi_mean"]
        row["cmi_std"]  = stability["cmi_std"]
        row["cmi_min"]  = stability["cmi_min"]
        row["cmi_max"]  = stability["cmi_max"]
        all_community_summaries.append(row)
        global_community_index += 1

community_summary_df = (
    pd.DataFrame(all_community_summaries)
    .sort_values(["micro_cluster_id", "community_index"])
    .reset_index(drop=True)
)

community_summary_df.to_csv(ROOT_LEIDEN_MICRO_DIR / "community_summary_micro.csv.gz", index=False, compression="gzip")
print(f"sav {len(community_summary_df)} communities to community_summary_micro.csv.gz")

Saved 44268 communities to community_summary.csv
